In [ ]:
# | default_exp gsc_insights

In [ ]:
# | export
from datetime import datetime, timedelta


In [ ]:
# | export


def apply_strftime(date: datetime):
    return date.strftime("%Y-%m-%d")


def get_date_ranges_for_comparison(days: int = 30  # Number of days per period
                                   ) -> tuple[tuple[str, str], tuple[str, str]]:
    "Return two consecutive date ranges for period comparison, accounting for 3-day GSC delay."
    fmt = lambda d: d.strftime("%Y-%m-%d")
    latest = datetime.now() - timedelta(days=3)
    period_a_end = latest
    period_a_start = latest - timedelta(days=days)
    period_b_end = period_a_start
    period_b_start = period_b_end - timedelta(days=days)
    return (fmt(period_a_start), fmt(period_a_end)), (fmt(period_b_start), fmt(period_b_end))

In [ ]:
# | hide
#| eval: false
get_date_ranges_for_comparison(30)

(('2026-03-02', '2026-04-01'), ('2026-01-31', '2026-03-02'))

In [ ]:
# | export
from sqlmodel import Session
from seo_rat.gsc_storage import *
from seo_rat.sqlite_db import get_session

In [ ]:
# | export
def compare_periods(recent: list[dict],  # Recent period query data
                    previous: list[dict]  # Previous period query data
                    ) -> list[dict]:
    "Compare two periods of query data and return trend analysis sorted by impression change."
    prev_by_query = {r["query"]: r for r in previous}
    trends = []
    for r in recent:
        p = prev_by_query.get(r["query"], {"total_clicks": 0, "total_impressions": 0, "avg_position": 100})
        click_change = r["total_clicks"] - p["total_clicks"]
        impression_change = r["total_impressions"] - p["total_impressions"]
        position_change = p["avg_position"] - r["avg_position"]
        trends.append({"query": r["query"],
                       "recent_clicks": r["total_clicks"], "prev_clicks": p["total_clicks"],
                       "click_change": click_change,
                       "recent_impressions": r["total_impressions"], "prev_impressions": p["total_impressions"],
                       "impression_change": impression_change,
                       "recent_position": r["avg_position"], "prev_position": p["avg_position"],
                       "position_change": position_change,
                       "trend": "rising" if impression_change > 0 and position_change > 0
                       else "declining" if impression_change < 0 or position_change < 0
                       else "stable"})
    return sorted(trends, key=lambda t: t["impression_change"], reverse=True)


In [ ]:
# | export
def detect_query_trends(session: Session,  # Active database session
                        site_url: str,  # GSC property URL
                        page_path: str | None = None,  # Filter by page path substring
                        days: int = 30,  # Days per comparison period
                        limit: int = 100  # Max queries to analyze
                        ) -> list[dict]:
    "Detect rising and declining queries by comparing two consecutive periods."
    (recent_start, recent_end), (prev_start, prev_end) = get_date_ranges_for_comparison(days)
    return compare_periods(
        get_top_queries(session, site_url, recent_start, recent_end, page_path=page_path, limit=limit),
        get_top_queries(session, site_url, prev_start, prev_end, page_path=page_path, limit=limit))


In [ ]:
# | hide
#| eval: false

session = get_session()

trends = detect_query_trends(
    session,
    site_url="sc-domain:kareemai.com",
    days=30,
    limit=50,
)
for t in trends[:10]:
    print(f"{t['trend']:>9} | imp: {t['impression_change']:+6d} | pos: {t['position_change']:+6.1f} | {t['query']}")




   rising | imp:   +681 | pos:  +92.5 | huawei freebuds 5i
declining | imp:   +281 | pos:   -0.9 | huawei freebuds 7i reviews
   rising | imp:   +207 | pos:  +95.5 | colgrep
   rising | imp:   +132 | pos:  +90.4 | oneplus pad 3
declining | imp:    +33 | pos:   -0.4 | huawei freebuds 7i anc review
   rising | imp:    +15 | pos:  +88.9 | huawei freebuds 7i รีวิว
   rising | imp:    +14 | pos:  +10.3 | one plus pad 3
   rising | imp:    +14 | pos:  +96.8 | huawei freebuds 7 review
   rising | imp:     +7 | pos:   +0.1 | huawei freebuds 5i vs 6i vs 7i
   rising | imp:     +6 | pos:  +88.8 | ticktick vs super productivity


In [ ]:
# | hide
#| eval: false
page_trends = detect_query_trends(
    session,
    site_url="sc-domain:kareemai.com",
    page_path="super_productivity_app",
    days=30,
    limit=50,
)
for t in page_trends[:10]:
    print(f"{t['trend']:>9} | imp: {t['impression_change']:+6d} | pos: {t['position_change']:+6.1f} | {t['query']}")

   rising | imp:    +47 | pos:   +7.2 | super productivity
declining | imp:    +18 | pos:   -2.4 | super productivity vs todoist
   rising | imp:    +16 | pos:  +18.4 | super productivity app
   rising | imp:    +13 | pos:   +1.6 | super productivity app features 2026
   rising | imp:    +12 | pos:   +0.8 | super productivity app features
   rising | imp:    +11 | pos:  +85.6 | superproductivity
   rising | imp:    +10 | pos:  +93.2 | super productivity features
   rising | imp:     +9 | pos:  +93.8 | super productivity vs ticktick
   rising | imp:     +9 | pos:  +92.2 | 32afd03550b1fac68ddb613598fd0643a00d1d99 super-productivity user-input.service.ts
   rising | imp:     +8 | pos:  +95.1 | todoist vs super productivity


In [ ]:
# | export
INTENT_PATTERNS = {
    "informational": [
        "what is",
        "how to",
        "guide",
        "explain",
        "tutorial",
        "ما هو",
        "كيف",
        "شرح",
        "تعريف",
        "دليل",
    ],
    "comparison": [
        "vs",
        "alternative",
        "best",
        "compare",
        "difference",
        "مقارنة",
        "أفضل",
        "بديل",
        "الفرق بين",
        "ضد",
    ],
    "transactional": [
        "buy",
        "price",
        "review",
        "specs",
        "discount",
        "سعر",
        "شراء",
        "مراجعة",
        "مواصفات",
        "طلب",
    ],
    "commercial": [
        "pricing",
        "cheapest",
        "affordable",
        "top rated",
        "comparison",
        "worth it",
        "cost",
        "budget",
        "rental",
        "أسعار",
        "أرخص",
        "تكلفة",
        "ميزانية",
        "إيجار",
    ],
    "navigational": [
        "site:",
        "login",
        "official",
        "website",
        "github",
        "موقع",
        "تسجيل دخول",
        "الرسمي",
    ],
}


In [ ]:
# | export
def classify_intent(query: str  # Search query
                    ) -> str:
    "Classify query intent based on keyword patterns."
    for intent, patterns in INTENT_PATTERNS.items():
        if any(p in query.lower() for p in patterns): return intent
    return "informational"

In [ ]:
#| hide
test_list = [
    "super productivity review",
    "super productivity app review",
    "super productivity todo app features",
    "super productivity",
    "super productivity app features",
    "super productive",
    "super productivity vs todoist",
    "super productivity docker compose",
    "is it completely free?",
    "super productivity gantt chart",
    "superproductivity app",
    "super productivity",
    "how to use super productivity",
]

In [ ]:
#| hide
[classify_intent(test) for test in test_list]

['transactional',
 'transactional',
 'informational',
 'informational',
 'informational',
 'informational',
 'comparison',
 'informational',
 'informational',
 'informational',
 'informational',
 'informational',
 'informational']

In [ ]:
# | export


def classify_page_intents(session: Session,  # Active database session
                          site_url: str,  # GSC property URL
                          start_date: str,  # Start date (YYYY-MM-DD)
                          end_date: str,  # End date (YYYY-MM-DD)
                          country: str | None = None,  # Filter by country code
                          page_path: str | None = None,  # Filter by page path substring
                          limit: int = 10  # Max rows to return
                          ) -> list[dict]:
    "Get top queries with intent classification."
    return [{**row, "intent": classify_intent(row["query"])}
            for row in get_top_queries(session, site_url, start_date, end_date,
                                       country=country, page_path=page_path, limit=limit)]



In [ ]:
# | hide
#| eval: false
page_intents = classify_page_intents(
    session,
    site_url="sc-domain:kareemai.com",
    page_path="gpuvec",
    limit=50,
    start_date=2,
    end_date=300,
)
page_intents


[{'query': 'cloud gpu pricing',
  'total_clicks': 4,
  'total_impressions': 2326,
  'avg_position': 56.05561004273504,
  'avg_ctr': 0.0021527777777777778,
  'intent': 'commercial'},
 {'query': 'cloud gpu pricing comparison',
  'total_clicks': 1,
  'total_impressions': 360,
  'avg_position': 32.543224043715846,
  'avg_ctr': 0.001639344262295082,
  'intent': 'commercial'},
 {'query': 'yes',
  'total_clicks': 0,
  'total_impressions': 1,
  'avg_position': 4.0,
  'avg_ctr': 0.0,
  'intent': 'informational'},
 {'query': 'which cloud platform has the best hourly gpu rates?',
  'total_clicks': 0,
  'total_impressions': 2,
  'avg_position': 82.0,
  'avg_ctr': 0.0,
  'intent': 'comparison'},
 {'query': 'which cloud gpu provider offers the best price per hour?',
  'total_clicks': 0,
  'total_impressions': 1,
  'avg_position': 13.0,
  'avg_ctr': 0.0,
  'intent': 'comparison'},
 {'query': 'which cloud gpu provider has the most flexible pricing options',
  'total_clicks': 0,
  'total_impressions': 

In [ ]:
# | export
STOP_WORDS = {
    "a",
    "an",
    "the",
    "is",
    "it",
    "in",
    "to",
    "for",
    "of",
    "and",
    "or",
    "how",
    "what",
    "who",
    "which",
    "this",
    "that",
    "هل",
    "في",
    "من",
    "على",
    "هو",
    "هذا",
    "ما",
}


def query_words_in_content(query: str,  # Search query
                           content: str  # Page content
                           ) -> bool:
    "Check if all significant words from a query appear in the content."
    words = [w for w in query.lower().split() if w not in STOP_WORDS and len(w) > 1]
    return all(w in content.lower() for w in words)


In [ ]:
# | export
def find_missing_queries(queries: list[dict],  # GSC query dicts
                         content: str  # Page content
                         ) -> list[dict]:
    "Find GSC queries whose significant words don't appear in the page content."
    return [q for q in queries if not query_words_in_content(q["query"], content)]


In [ ]:
#| hide
#| eval: false
from dotenv import load_dotenv
import os
from seo_rat.content_parser import get_page_content

from seo_rat.gsc_client import get_date_range
from seo_rat.index_tracking import fetch_sitemap_urls
from seo_rat.content_mapper import map_all_urls_to_files

load_dotenv()
BASE_PATH = os.environ["SEO_RAT_BASE_PATH"]

urls = fetch_sitemap_urls("https://kareemai.com/sitemap.xml")
url_mapping = map_all_urls_to_files(
     BASE_PATH,
    "https://kareemai.com/",
    urls,
)


In [ ]:
# | hide
#| eval: false
start, end = get_date_range("last_days", days=120)
page_queries = get_top_queries(
    session,
    site_url="sc-domain:kareemai.com",
    start_date=start,
    end_date=end,
    page_path="gpuvec",
    limit=5000,
)

# Get content from file
file_path = url_mapping[
    "https://kareemai.com/blog/posts/life_style/gpuvec.html"
]
content = get_page_content(file_path, is_quarto=True)

# Find missing queries
missing = find_missing_queries(page_queries, content)

print(f"Total queries: {len(page_queries)}")
print(f"Missing from content: {len(missing)}\n")
for m in missing:
    print(
        f"  {m['query']:50} | imp: {m['total_impressions']} | pos: {m['avg_position']:.1f}"
    )


Total queries: 85
Missing from content: 15

  yes                                                | imp: 1 | pos: 4.0
  rent gpu hivenet vs aws                            | imp: 1 | pos: 49.0
  nvidia l20 48gb cloud instance price per hour china 2025 | imp: 1 | pos: 10.0
  huawei cloud gpu instances specifications pricing 2024 | imp: 7 | pos: 4.6
  hivenet rtx 4090 vs azure                          | imp: 2 | pos: 34.5
  hivenet nvidia gpu vs google cloud                 | imp: 5 | pos: 41.3
  hivenet cheap gpu rental vs google cloud           | imp: 1 | pos: 12.0
  hivenet cheap gpu rental vs aws                    | imp: 3 | pos: 16.7
  gpu comparsion                                     | imp: 1 | pos: 8.0
  get hivenet gpu vs aws                             | imp: 3 | pos: 33.3
  fast hivenet gpu vs aws                            | imp: 1 | pos: 33.0
  compare gpu cloud services                         | imp: 2 | pos: 89.5
  cloud gpu prices                                   | imp: 1

In [ ]:
#| export 
def find_green_keywords(session: Session,  # Active database session
                        site_url: str,  # GSC property URL
                        page_path: str,  # Page path to analyze
                        content: str,  # Page content
                        days: int = 30,  # Days per comparison period
                        limit: int = 100  # Max queries to analyze
                        ) -> list[dict]:
    "Find emerging queries not yet covered in page content."
    trends = detect_query_trends(session, site_url, page_path=page_path, days=days, limit=limit)
    rising = [t for t in trends if t["trend"] == "rising" and t["prev_impressions"] <= 5]
    return find_missing_queries(rising, content)


In [ ]:
#| hide
#| eval: false
file_path = url_mapping["https://kareemai.com/blog/posts/life_style/gpuvec.html"]
content = get_page_content(file_path, is_quarto=True)

green = find_green_keywords(
    session,
    site_url="sc-domain:kareemai.com",
    page_path="gpuvec",
    content=content,
    days=90,
)

print(f"Green keywords found: {len(green)}\n")
for g in green:
    print(
        f"  {g['query']:50} | imp: {g['prev_impressions']} → {g['recent_impressions']} | pos: {g['recent_position']:.1f}")


Green keywords found: 12

  huawei cloud gpu instances specifications pricing 2024 | imp: 0 → 7 | pos: 4.6
  hivenet nvidia gpu vs google cloud                 | imp: 0 → 5 | pos: 41.3
  buy hivenet compute benchmark                      | imp: 0 → 4 | pos: 12.8
  hivenet cheap gpu rental vs aws                    | imp: 0 → 3 | pos: 16.7
  get hivenet gpu vs aws                             | imp: 0 → 3 | pos: 33.3
  hivenet rtx 4090 vs azure                          | imp: 0 → 2 | pos: 34.5
  buy hivenet cloud                                  | imp: 0 → 2 | pos: 50.5
  yes                                                | imp: 0 → 1 | pos: 4.0
  rent gpu hivenet vs aws                            | imp: 0 → 1 | pos: 49.0
  hivenet cheap gpu rental vs google cloud           | imp: 0 → 1 | pos: 12.0
  gpu comparsion                                     | imp: 0 → 1 | pos: 8.0
  fast hivenet gpu vs aws                            | imp: 0 → 1 | pos: 33.0
